In [1]:
!pip install transformers datasets accelerate sentencepiece underthesea -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 63.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModel,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling
)

from torch.utils.data import DataLoader

from datasets import load_dataset


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)


cuda


đồng bộ hóa ngôn ngữ giữa mô hình cũ (Teacher) và mô hình mới (Student), đồng thời cung cấp con số kỹ thuật để bạn xây dựng "cái khung" mô hình Student không bị lỗi.


In [4]:
teacher_name = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(
    teacher_name
)

print(tokenizer.vocab_size)


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

64000


thiết lập mô hình PhoBERT làm một "cuốn từ điển sống" cố định chỉ dùng nó để lấy kết quả chuẩn, sau đó ép mô hình Student phải tạo ra kết quả giống như vậy.


In [5]:
teacher = AutoModel.from_pretrained(
    teacher_name
).to(device)

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

"bản sao thu nhỏ" của PhoBERT. Hiện tại, mô hình này mới chỉ là một cái khung với các con số ngẫu nhiên. Nó chưa biết tiếng Việt và chưa có tri thức gì cả.


In [6]:
student_config = RobertaConfig(
    vocab_size=tokenizer.vocab_size + 1, 
    hidden_size=768,
    num_hidden_layers=6,
    num_attention_heads=12,
    intermediate_size=3072,
    max_position_embeddings=514,
)

student = RobertaForMaskedLM(
    student_config
).to(device)


lấy "bộ não" 12 múi của Teacher, cắt ra 6 múi khỏe nhất để lắp vào Student.


In [7]:
teacher_layers = teacher.encoder.layer
student_layers = student.roberta.encoder.layer

selected_layers = [0, 2, 4, 6, 8, 10]

for s_idx, t_idx in enumerate(selected_layers):

    student_layers[s_idx].load_state_dict(
        teacher_layers[t_idx].state_dict()
    )

print("Copied teacher layers -> student")


Copied teacher layers -> student


In [8]:

full_dataset = load_dataset("ithieund/VietNews-Abs-Sum")


dataset = full_dataset["train"].shuffle(seed=42).select(range(10000))



README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


processed/train.tsv:   0%|          | 0.00/325M [00:00<?, ?B/s]

processed/train_desegmented.tsv:   0%|          | 0.00/325M [00:00<?, ?B/s]

raw/train.tsv:   0%|          | 0.00/349M [00:00<?, ?B/s]

...

processed/test_desegmented.tsv:   0%|          | 0.00/74.5M [00:00<?, ?B/s]

raw/test.tsv:   0%|          | 0.00/75.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/303686 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/67010 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/67640 [00:00<?, ? examples/s]

In [9]:
print(dataset)

print(dataset[0])


Dataset({
    features: ['guid', 'title', 'abstract', 'article'],
    num_rows: 10000
})
{'guid': 93259, 'title': 'Cá hô 130 kg được thương lái Sài Gòn mua', 'abstract': 'Sáng 7/12 , con cá hô khủng do một ngư dân Vĩnh Long bắt đã được thương lái mua với giá khoảng 200 triệu đồng và chuyển về Sài Gòn .', 'article': 'Con cá hô dài khoảng 1,8 m , nặng gần 130 kg . Khoảng 15h ngày 6/12 , ngư dân Nguyễn Thanh Lộc ( 50 tuổi ) và con trai là Nguyễn Văn Minh ( 27 tuổi , ngụ huyện Mang Thít , tỉnh Vĩnh Long ) đánh bắt cá trên sông Cổ Chiên . Cả hai cha con hốt hoảng khi một con cá hô lớn mắc lưới . Với sự giúp sức của nhiều người , anh Minh đã đưa cá lên bờ . Con cá dài 1,8 m , nặng gần 130 kg . Nhiều thương lái đã đến trả giá mua cá , cuối cùng gia đình anh Minh đã bán cho một hệ thống nhà hàng ở TP HCM , với giá gần 200 triệu đồng . Sau khi được đông lạnh , sáng nay con cá hô được vận chuyển về TP HCM và xẻ thịt để bán cho các nhà hàng . Sáng nay con cá hô được chuyển về nhà hàng ở TP HCM . 

input đi vào mô hình đều có cấu trúc đồng nhất, gọn gàng và đúng định dạng mà mô hình Student yêu cầu


In [10]:
max_length = 256

def tokenize_function(examples):

    return tokenizer(
        examples["article"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )


biến tokenized_dataset sẽ chứa dữ liệu ở dạng các "ma trận số" sẵn sàng để nạp vào GPU.


In [11]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

DataCollator giống như một giáo viên liên tục dùng bút xóa che đi các từ trong bài báo để bắt Student phải suy luận


In [12]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


In [13]:
train_loader = DataLoader(
    tokenized_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)


lọc bỏ "rác" (padding) và cô đọng ý nghĩa của hàng trăm từ về thành một bộ số (vector) duy nhất cực kỳ mạnh mẽ.


In [14]:
def mean_pooling(hidden_states, attention_mask):

    mask = attention_mask.unsqueeze(-1)

    masked_hidden = hidden_states * mask

    return (
        masked_hidden.sum(dim=1)
        / mask.sum(dim=1)
    )


hệ thống đánh giá đa chiều cho Student:

Học vẹt (MSE): Bắt chước từng giá trị vector của thầy.

Học hiểu (Cosine): Bắt chước ý nghĩa/hướng của câu văn.

Tự học (MLM - ở bước sau): Tự đoán từ bị che để hiểu ngữ pháp.

Sự kết hợp này giúp Student vừa giữ được kiến thức của PhoBERT, vừa có khả năng tạo ra các vector Embedding cực kỳ sắc bén cho việc Tóm tắt trích xuất


In [15]:
optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=5e-5
)

mse_loss_fn = nn.MSELoss()

cosine_loss_fn = nn.CosineEmbeddingLoss()


Tao thu muc luu ket qua huan luyen, model weights, bieu do va file test de nop bai hoac kiem tra lai de hon.


In [16]:
from pathlib import Path
import json
import shutil

artifact_dir = Path("results_distill_phobert")
artifact_dir.mkdir(parents=True, exist_ok=True)

plots_dir = artifact_dir / "plots"
reports_dir = artifact_dir / "reports"
samples_dir = artifact_dir / "sample_outputs"
weights_dir = artifact_dir / "model_weights"

for folder in [plots_dir, reports_dir, samples_dir, weights_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Results will be saved to: {artifact_dir.resolve()}")


Results will be saved to: /kaggle/working/results_distill_phobert


Vòng lặp này đang "gọt giũa" mô hình 6 lớp sao cho nó vừa biết đoán từ, vừa có cách nhìn nhận ý nghĩa câu văn y hệt như mô hình PhoBERT 12 lớp bản gốc.


In [ ]:
# --- STEP 1: Helper function ---
def get_embedding(text, model, tokenizer, device):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        # Use model.roberta to get hidden states
        outputs = model.roberta(**inputs)

    hidden = outputs.last_hidden_state
    embedding = mean_pooling(hidden, inputs["attention_mask"])
    return embedding

# --- STEP 2: Training loop + loss history ---
epochs = 3
training_history = []
student.train()

for epoch in range(epochs):
    total_loss_epoch = 0
    total_mlm_epoch = 0
    total_distill_epoch = 0
    total_cosine_epoch = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad():
            teacher_outputs = teacher(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            teacher_hidden = teacher_outputs.hidden_states[-1]

        student_outputs = student(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )
        student_hidden = student_outputs.hidden_states[-1]

        mlm_loss = student_outputs.loss
        teacher_embed = mean_pooling(teacher_hidden, attention_mask)
        student_embed = mean_pooling(student_hidden, attention_mask)

        teacher_embed = nn.functional.normalize(teacher_embed, dim=-1)
        student_embed = nn.functional.normalize(student_embed, dim=-1)

        distill_loss = mse_loss_fn(student_embed, teacher_embed)
        target = torch.ones(input_ids.size(0)).to(device)
        cosine_loss = cosine_loss_fn(student_embed, teacher_embed, target)

        total_loss = (0.4 * mlm_loss + 0.4 * distill_loss + 0.2 * cosine_loss)

        total_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss_epoch += total_loss.item()
        total_mlm_epoch += mlm_loss.item()
        total_distill_epoch += distill_loss.item()
        total_cosine_epoch += cosine_loss.item()

    num_batches = len(train_loader)
    epoch_log = {
        "epoch": epoch + 1,
        "total_loss": total_loss_epoch / num_batches,
        "mlm_loss": total_mlm_epoch / num_batches,
        "distill_mse_loss": total_distill_epoch / num_batches,
        "cosine_loss": total_cosine_epoch / num_batches,
    }
    training_history.append(epoch_log)

    print(
        f"Epoch {epoch + 1} | "
        f"Total: {epoch_log['total_loss']:.4f} | "
        f"MLM: {epoch_log['mlm_loss']:.4f} | "
        f"MSE: {epoch_log['distill_mse_loss']:.4f} | "
        f"Cosine: {epoch_log['cosine_loss']:.4f}"
    )


Luu lich su huan luyen ra CSV/JSON va ve do thi loss de xem model hoc co on dinh khong.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(training_history)
history_df.to_csv(reports_dir / "training_history.csv", index=False, encoding="utf-8-sig")
history_df.to_json(reports_dir / "training_history.json", orient="records", force_ascii=False, indent=2)

plt.figure(figsize=(9, 5))
for col in ["total_loss", "mlm_loss", "distill_mse_loss", "cosine_loss"]:
    plt.plot(history_df["epoch"], history_df[col], marker="o", linewidth=2, label=col)

plt.title("Training Loss History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(history_df["epoch"])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(plots_dir / "training_loss_curve.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved training history:", (reports_dir / "training_history.csv").resolve())
print("Saved training plot:", (plots_dir / "training_loss_curve.png").resolve())


In [ ]:
save_path = weights_dir

try:
    student.save_pretrained(save_path, safe_serialization=True)
except TypeError:
    student.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

model_info = {
    "teacher_name": teacher_name,
    "student_architecture": "RobertaForMaskedLM",
    "num_hidden_layers": student_config.num_hidden_layers,
    "hidden_size": student_config.hidden_size,
    "num_attention_heads": student_config.num_attention_heads,
    "intermediate_size": student_config.intermediate_size,
    "max_position_embeddings": student_config.max_position_embeddings,
    "vocab_size": student_config.vocab_size,
    "selected_teacher_layers": selected_layers,
    "max_length": max_length,
    "train_samples": len(dataset),
    "epochs": epochs,
}

with open(artifact_dir / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(str(weights_dir), "zip", root_dir=weights_dir)

print("Saved model weights to:", save_path.resolve())
print("Saved zipped weights to:", Path(zip_path).resolve())
print("Saved model info to:", (artifact_dir / "model_info.json").resolve())


In [ ]:
from transformers import RobertaForMaskedLM

model = RobertaForMaskedLM.from_pretrained(
    weights_dir
).to(device)


biến "chữ viết" khô khan thành "tọa độ không gian". Trong không gian đó, những câu có ý nghĩa giống nhau sẽ nằm gần nhau, giúp máy tính "hiểu" và tóm tắt văn bản một cách thông minh.


In [ ]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():

        outputs = model.roberta(
            **inputs
        )

    hidden = outputs.last_hidden_state

    embedding = mean_pooling(
        hidden,
        inputs["attention_mask"]
    )

    return embedding


In [ ]:
s1 = "Tui thích học NLP"
s2 = "Tui Yêu học NLP"

e1 = get_embedding(s1)
e2 = get_embedding(s2)

sim = torch.cosine_similarity(
    e1,
    e2
)

print(sim.item())


In [ ]:
!pip install underthesea


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from underthesea import sent_tokenize

# 1. Hàm Mean Pooling (Bắt buộc phải có)
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# 2. Hàm Get Embedding (Định nghĩa chuẩn 4 tham số)
def get_embedding(text, model, tokenizer, device):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        # Dùng model.roberta để lấy output từ các lớp Transformer
        outputs = model.roberta(**inputs)

    hidden = outputs.last_hidden_state
    embedding = mean_pooling(hidden, inputs["attention_mask"])
    return embedding

# 3. Hàm Tóm tắt (Đã chỉnh sửa để gọi đúng tham số)
def summarize_long_text(text, num_sentences=3):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    sentence_embeddings = []
    for sent in sentences:
        # Gọi hàm với đúng 4 tham số như định nghĩa ở trên
        embed = get_embedding(sent, student, tokenizer, device)
        sentence_embeddings.append(embed)

    sentence_embeddings = torch.cat(sentence_embeddings, dim=0)
    article_centroid = torch.mean(sentence_embeddings, dim=0, keepdim=True)
    similarities = F.cosine_similarity(sentence_embeddings, article_centroid)

    top_indices = torch.topk(similarities, k=num_sentences).indices.tolist()
    top_indices.sort()

    summary = " ".join([sentences[i] for i in top_indices])
    return summary


In [ ]:
van_ban = """ 
Trong bài học hôm nay, cô sẽ trình bày về quá trình phát triển của các mô hình xử lý ngôn ngữ tự nhiên, những hạn chế của phương pháp học có giám sát truyền thống và lý do vì sao mô hình ngôn ngữ lớn (LLM) ra đời. Nội dung này được trình bày dựa trên transcript bài giảng trong file đã cung cấp.

Đầu tiên, để xây dựng một hệ thống xử lý ngôn ngữ tự nhiên theo cách truyền thống, các bạn phải giải quyết bài toán dữ liệu. Dữ liệu trong NLP không giống như dữ liệu hình ảnh hay dữ liệu số thông thường. Các bạn phải chuẩn bị một bộ câu hỏi và câu trả lời tương ứng, tức là dữ liệu đầu vào X và đầu ra Y. Sau đó, các bạn phải tiến hành gán nhãn cho toàn bộ dữ liệu này. Tuy nhiên, khó khăn nằm ở chỗ cùng một ý nghĩa nhưng con người có thể diễn đạt bằng rất nhiều cách khác nhau. Ví dụ như câu “Bạn đang ở đâu?” và “Bạn đang ở trường à?” đều có thể mang ý nghĩa liên quan đến vị trí của người nói. Điều này khiến việc gán nhãn trong NLP khó khăn hơn rất nhiều so với xử lý ảnh hay dữ liệu có cấu trúc.

Trong xử lý ngôn ngữ tự nhiên, dữ liệu không cố định mà thay đổi liên tục theo thời gian. Hôm nay người ta nói theo cách này, nhưng ngày mai có thể xuất hiện một cách diễn đạt khác hoàn toàn. Vì vậy, nếu mô hình chỉ học trên một tập dữ liệu cố định thì nó sẽ trở nên cứng nhắc và không thể thích nghi với dữ liệu mới. Đây chính là hạn chế lớn của phương pháp học có giám sát truyền thống. Mô hình chỉ trả lời tốt những gì đã được học trước đó, còn khi gặp thông tin mới hoặc cách diễn đạt mới thì rất dễ trả lời sai hoặc không tự nhiên.

Một vấn đề khác là việc gán nhãn dữ liệu rất tốn công sức. Nếu ngày mai xuất hiện thêm dữ liệu mới thì không thể nào con người cứ tiếp tục ngồi gán nhãn mãi được. Khi dữ liệu ngày càng lớn, phương pháp học có giám sát truyền thống trở nên không còn phù hợp nữa. Chính vì vậy, các công ty công nghệ lớn bắt đầu đặt ra câu hỏi: làm thế nào để xây dựng một mô hình có thể học từ dữ liệu mới mà không cần phụ thuộc hoàn toàn vào dữ liệu được gán nhãn?

Từ đó, người ta bắt đầu tìm đến hướng tiếp cận học tăng cường (Reinforcement Learning). Ý tưởng của học tăng cường là mô hình không chỉ học từ dữ liệu ban đầu mà còn học từ kinh nghiệm và phản hồi của môi trường. Khi gặp dữ liệu mới, mô hình sẽ dựa trên kiến thức trước đó để đưa ra quyết định phù hợp hơn. Điều này giúp mô hình có khả năng thích nghi tốt hơn với sự thay đổi liên tục của ngôn ngữ.

Song song với sự phát triển của thuật toán, phần cứng máy tính cũng ngày càng mạnh hơn. Trước đây, do hạn chế về tài nguyên tính toán nên người ta chỉ có thể sử dụng các mô hình nhỏ hoặc vừa phải. Nhưng khi GPU và hạ tầng tính toán phát triển mạnh, các công ty bắt đầu nghĩ đến việc xây dựng những mô hình lớn hơn với lượng dữ liệu khổng lồ. Mục tiêu là tạo ra mô hình có khả năng hiểu và sinh ngôn ngữ tự nhiên một cách linh hoạt hơn.

Trong quá trình đó, Transformer được xem là một bước đột phá lớn của NLP. Khác với các mô hình như LSTM vốn tập trung học theo chuỗi tuần tự, Transformer sử dụng cơ chế Attention để nhìn được mối quan hệ tổng quát giữa các từ trong câu. Điều này giúp mô hình xử lý dữ liệu hiệu quả hơn và học được nhiều ngữ cảnh hơn. Tuy nhiên, Transformer chỉ phát huy hiệu quả thực sự khi được huấn luyện trên tập dữ liệu rất lớn.

Giảng viên cũng nhắc đến quá trình phát triển của các mô hình ngôn ngữ lớn như ChatGPT. Theo bài giảng, OpenAI đã dành nhiều năm để xây dựng bộ dữ liệu khổng lồ từ nhiều lĩnh vực khác nhau. Nhờ có lượng dữ liệu lớn cùng phần cứng mạnh, họ có thể tinh chỉnh kiến trúc Transformer để tạo ra những mô hình vừa mạnh vừa tối ưu hơn. Những mô hình này có thể trả lời được rất nhiều loại câu hỏi khác nhau vì đã được học từ lượng thông tin khổng lồ trên nhiều lĩnh vực.

Tuy nhiên, chỉ có dữ liệu lớn thôi vẫn chưa đủ. Nếu mô hình chỉ trả lời đúng theo dữ liệu đã học thì câu trả lời vẫn sẽ cứng nhắc và thiếu tự nhiên. Vì vậy, người ta kết hợp thêm học tăng cường để mô hình có thể cải thiện phản hồi dựa trên trải nghiệm thực tế.

Để giải thích học tăng cường, giảng viên đưa ra ví dụ về một đứa trẻ. Ban đầu, đứa trẻ được cha mẹ dạy cách trả lời một câu hỏi nào đó. Ví dụ khi được hỏi “Huy có đẹp trai không?”, cha mẹ dạy rằng câu trả lời là “không”. Khi đứa trẻ trả lời như vậy, nó nhận lại phản ứng không vui từ bạn Huy và những người xung quanh. Từ trải nghiệm đó, nó bắt đầu học được rằng mặc dù câu trả lời đúng với kiến thức ban đầu nhưng cách trả lời lại không phù hợp với môi trường.

Ở lần tiếp theo, đứa trẻ sẽ thay đổi cách trả lời. Nó vẫn giữ nội dung tương tự nhưng diễn đạt nhẹ nhàng và khéo léo hơn để người nghe cảm thấy vui vẻ. Đây chính là cách học tăng cường hoạt động: mô hình không chỉ dựa vào kiến thức ban đầu mà còn dựa vào phản hồi từ môi trường để điều chỉnh hành động trong tương lai. Qua nhiều lần tương tác, mô hình sẽ học được cách tạo ra câu trả lời tự nhiên, linh hoạt và phù hợp hơn với mục tiêu giao tiếp.




"""


In [ ]:

# Thử tóm tắt lấy 3 câu
ket_qua = summarize_long_text(van_ban, num_sentences=10)
print("BẢN TÓM TẮT CUỐI CÙNG:")
print(ket_qua)


In [ ]:
# Lấy 100 mẫu từ tập test
test_dataset = full_dataset["test"].shuffle(seed=42).select(range(100))

# Xem thử mẫu đầu tiên
print(test_dataset[0])


In [ ]:
print(full_dataset["train"].features.keys())


In [ ]:
# Chạy thử nghiệm trên 5 mẫu đầu tiên từ 100 mẫu đã chọn
for i in range(5): 
    # Cập nhật tên cột đúng: 'article' và 'abstract'
    article_text = test_dataset[i]["article"] 
    reference_summary = test_dataset[i]["abstract"] 
    
    # Gọi hàm tóm tắt của bạn
    generated_summary = summarize_long_text(article_text, num_sentences=3)
    
    print(f"--- Mẫu số {i+1} ---")
    print(f"TIÊU ĐỀ: {test_dataset[i]['title']}") # In thêm tiêu đề cho rõ ràng
    print(f"\nTÓM TẮT GỐC (Abstract):\n{reference_summary}")
    print(f"\nAI TRÍCH XUẤT (Extractive):\n{generated_summary}")
    print("="*80)


In [ ]:
!pip install rouge-score


In [ ]:
from rouge_score import rouge_scorer
import pandas as pd
from tqdm import tqdm

def evaluate_model(dataset, num_samples=100):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    results = []

    print(f"Đang đánh giá trên {num_samples} mẫu...")
    
    # Duyệt qua các mẫu trong dataset
    for i in tqdm(range(num_samples)):
        article = dataset[i]["article"]
        reference = dataset[i]["abstract"]
        
        # 1. Sinh bản tóm tắt từ model của bạn
        try:
            generated = summarize_long_text(article, num_sentences=3)
        except:
            continue # Bỏ qua nếu có lỗi xử lý văn bản
            
        # 2. Tính điểm ROUGE
        scores = scorer.score(reference, generated)
        
        results.append({
            'rouge1': scores['rouge1'].fmeasure,
            'rouge2': scores['rouge2'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        })

    # 3. Tổng hợp kết quả trung bình
    df_results = pd.DataFrame(results)
    avg_scores = df_results.mean()

    print("\n" + "="*30)
    print("KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH")
    print("="*30)
    print(f"ROUGE-1: {avg_scores['rouge1']:.4f} (Độ khớp từ đơn)")
    print(f"ROUGE-2: {avg_scores['rouge2']:.4f} (Độ khớp cụm 2 từ)")
    print(f"ROUGE-L: {avg_scores['rougeL']:.4f} (Độ dài chuỗi khớp nhất)")
    print("="*30)
    
    return avg_scores

# Chạy đánh giá
# Lưu ý: Đảm bảo test_dataset đã được load 100 mẫu như bước trước
final_metrics = evaluate_model(test_dataset, num_samples=100)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from rouge_score import rouge_scorer
import pandas as pd
from tqdm import tqdm
import textwrap


def save_summary_image(record, output_path):
    title = record.get("title") or f"Sample {record['sample_id']}"
    generated = record["generated_summary"]
    reference = record["reference_summary"]

    fig, ax = plt.subplots(figsize=(12, 7))
    ax.axis("off")

    image_text = (
        f"TEST SAMPLE {record['sample_id']} - {title}\n\n"
        f"AI EXTRACTIVE SUMMARY:\n{textwrap.fill(generated, width=105)}\n\n"
        f"REFERENCE ABSTRACT:\n{textwrap.fill(reference, width=105)}\n\n"
        f"ROUGE-1: {record['rouge1']:.3f}   "
        f"ROUGE-2: {record['rouge2']:.3f}   "
        f"ROUGE-L: {record['rougeL']:.3f}"
    )

    ax.text(
        0.03, 0.97, image_text,
        va="top",
        ha="left",
        fontsize=11,
        linespacing=1.35,
        family="DejaVu Sans"
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def evaluate_and_export(dataset, num_samples=100, num_sentences=3, image_samples=5):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL", "rougeLsum"],
        use_stemmer=True
    )

    records = []
    for i in tqdm(range(num_samples)):
        article = dataset[i]["article"]
        reference = dataset[i]["abstract"]
        title = dataset[i].get("title", "")

        try:
            generated = summarize_long_text(article, num_sentences=num_sentences)
            scores = scorer.score(reference, generated)
        except Exception as exc:
            print(f"Skip sample {i}: {exc}")
            continue

        records.append({
            "sample_id": i + 1,
            "title": title,
            "reference_summary": reference,
            "generated_summary": generated,
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure,
            "rougeLsum": scores["rougeLsum"].fmeasure,
            "precision": scores["rougeL"].precision,
            "recall": scores["rougeL"].recall,
            "f1": scores["rougeL"].fmeasure,
        })

    results_df = pd.DataFrame(records)
    avg_scores = results_df[[
        "rouge1", "rouge2", "rougeL", "rougeLsum", "precision", "recall", "f1"
    ]].mean()

    results_csv = reports_dir / "test_results.csv"
    results_json = reports_dir / "test_results.json"
    metrics_json = reports_dir / "average_metrics.json"
    report_txt = reports_dir / "test_report.txt"

    results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
    results_df.to_json(results_json, orient="records", force_ascii=False, indent=2)

    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(avg_scores.to_dict(), f, ensure_ascii=False, indent=2)

    with open(report_txt, "w", encoding="utf-8") as f:
        f.write("KET QUA DANH GIA TOM TAT EXTRACTIVE\n")
        f.write("=" * 50 + "\n")
        f.write(f"So mau test: {len(results_df)}\n")
        f.write(f"So cau moi ban tom tat: {num_sentences}\n\n")
        for key, value in avg_scores.items():
            f.write(f"{key}: {value:.4f}\n")
        f.write("\nCAC MAU DAU TIEN\n")
        f.write("=" * 50 + "\n")
        for record in records[:image_samples]:
            f.write(f"\n--- Sample {record['sample_id']}: {record['title']} ---\n")
            f.write("AI summary:\n")
            f.write(record["generated_summary"] + "\n")
            f.write("Reference:\n")
            f.write(record["reference_summary"] + "\n")

    plt.figure(figsize=(10, 6))
    sns.set_style("whitegrid")
    metric_order = ["rouge1", "rouge2", "rougeL", "rougeLsum", "precision", "recall", "f1"]
    ax = sns.barplot(x=metric_order, y=avg_scores[metric_order].values, palette="muted")

    for p in ax.patches:
        ax.annotate(
            format(p.get_height(), ".3f"),
            (p.get_x() + p.get_width() / 2., p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 9),
            textcoords="offset points",
            fontsize=11,
            fontweight="bold"
        )

    plt.ylim(0, 1.0)
    plt.title("Evaluation Metrics - Extractive Summarization")
    plt.ylabel("Score")
    plt.xlabel("Metric")
    plt.xticks(rotation=35)
    plt.tight_layout()
    metrics_plot = plots_dir / "rouge_metrics_bar_chart.png"
    plt.savefig(metrics_plot, dpi=200, bbox_inches="tight")
    plt.show()

    for record in records[:image_samples]:
        image_path = samples_dir / f"sample_{record['sample_id']:02d}_summary.png"
        save_summary_image(record, image_path)

    print("Saved test CSV:", results_csv.resolve())
    print("Saved test JSON:", results_json.resolve())
    print("Saved average metrics:", metrics_json.resolve())
    print("Saved text report:", report_txt.resolve())
    print("Saved ROUGE chart:", metrics_plot.resolve())
    print("Saved sample images to:", samples_dir.resolve())

    return avg_scores, results_df


final_metrics, test_results_df = evaluate_and_export(
    test_dataset,
    num_samples=100,
    num_sentences=3,
    image_samples=5
)
final_metrics


Liet ke nhanh cac file ket qua da tao de biet nop hoac mo file nao.


In [ ]:
print("Generated result files:")
for path in sorted(artifact_dir.rglob("*")):
    if path.is_file():
        print("-", path.as_posix())
